# LSC-54 — descarga selectiva y conversión

`datos.json` pesa 55 GB, pero solo sirve una parte:

- las `rep_n` de un video son la misma grabación desplazada en x, así que se
  baja únicamente la `rep_0` de cada uno;
- la cara ocupa el 84 % de cada frame y el modelo no la usa.

El descargador recorre el archivo por rangos HTTP y solo lee lo que necesita.
Sale un `.jsonl` con el formato de `docs/06-data/model-input-format.md`.

**Por qué acá y no en local:** el servidor limita por IP. Desde una máquina de
casa cayó a 70 B/s después de unos 2 GB; Colab sale por otra IP y tiene mucho
más ancho de banda.

In [ ]:
!pip install -q requests numpy

REPO = 'https://github.com/mauro-1508/backendTS.git'
RAMA = 'feat/backend-dominios-ia'

import os
if not os.path.exists('download_lsc54.py'):
    !git clone -q --branch {RAMA} --depth 1 {REPO} repo
    !cp repo/services/recognition-service/src/ia/training/extraction/*.py .

print(sorted(f for f in os.listdir('.') if f.endswith('.py')))

## 1. Descarga

Cada tramo del archivo va en un hilo. **16 hilos es un punto razonable**: con
más, el servidor empieza a responder 429 y frena a todos. El avance se guarda
cada 30 s, así que si Colab corta la sesión, se vuelve a ejecutar esta celda y
sigue donde iba.

Se leen unos 13 GB para quedarse con ~11 GB de `rep_0`: el recorrido cuesta
alrededor del 3 % extra.

Para bajar solo algunas señas: `--solo bienvenido,baño`.

In [ ]:
# Drive se monta ANTES de bajar nada: la descarga escribe directo alli.
# El disco de Colab se borra cuando la sesion se desconecta, y con el se
# perderian horas de descarga.
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/traduce_senas'
SALIDA_CRUDA = f'{BASE}/datasets/lsc54_rep0'
SALIDA = f'{BASE}/datasets/lsc54.jsonl'

import os
os.makedirs(SALIDA_CRUDA, exist_ok=True)
print('la descarga se guarda en', SALIDA_CRUDA)

In [ ]:
!python download_lsc54.py --out {SALIDA_CRUDA} --workers 16


## 2. Conversión al formato del modelo

Renormaliza al marco de los hombros, descarta las manos que el dataset rellenó
copiando puntos del torso, recorta al tramo activo y deja 30 frames × 128
valores por clip.

In [ ]:
!python convert_lsc54.py --rep0 {SALIDA_CRUDA} --out {SALIDA}


## 3. Revisión

In [ ]:
import json
import numpy as np

recs = [json.loads(l) for l in open(SALIDA, encoding='utf-8')]
X = np.array([r['frames'] for r in recs], dtype=np.float32)

print('tensor:', X.shape, '(esperado: n x 30 x 128)')
print('señas:', len({r['sign_id'] for r in recs}), '| señantes:', len({r['signer'] for r in recs}))
print('banderas:', np.unique(X[:, :, 126:]), '(solo 0 y 1)')
print('manos vistas: %.0f%%' % (100 * X[:, :, 126:].mean()))

wr, wl = X[:, :, 0], X[:, :, 63]
print('x muñeca derecha %+.2f | izquierda %+.2f (signos opuestos)' % (
    np.nanmean(wr[np.abs(wr) > 1e-9]), np.nanmean(wl[np.abs(wl) > 1e-9])))

# Cuántas muestras quedaron por seña: las clases con muy pocas no sirven
# para entrenar y hay que sacarlas o volver a bajarlas.
from collections import Counter
cuenta = Counter(r['sign_id'] for r in recs)
print('\nseñas con menos de 10 muestras:', {k: v for k, v in cuenta.items() if v < 10})
print('mediana de muestras por seña:', int(np.median(list(cuenta.values()))))

## 4. Guardar en Drive

In [ ]:
# Ya esta todo en Drive; aqui solo se comprueba.
!ls -lh {BASE}/datasets/
print('
Si la sesion se cae, vuelve a ejecutar la celda de descarga: retoma sola.')